# Assignment 1

![](https://media.giphy.com/media/xT9C25UNTwfZuk85WP/giphy-downsized-large.gif)

Remember the rules of ~Fight~ Code Club:
1. ALWAYS DOCUMENT
2. Cite resources that you use (paste links)
3. Include the names people who you worked with
4. Be neat and organized

## Scrape and Clean data

Based on you proposal, scrape or collect your data:

1. One variable must be either: (40 pts)
    1. scraped from the web OR;
    2. collected from an API AND you must create one new variable that is "new" to the best of your knowledge (combination of other variables representing something new).

2. You must have at least 3 variables, but you may include as many as you want into you final dataset. Likely, you will want to include more to make graphs and regressions. (30pts)

3. You must *be able* to run a regression that makes some sense with this data (the regression doesn't have to be a complete model). Briefly describe one regression you would run with your variables. (**DO NOT** run a regression, yet). (15pts)

4. You must have one combined and cleaned dataset (15 pts)

You must submit one python notebook on how you scraped/gathered data from an api, and how you combined and cleaned you data. I should be able to run your code and reproduce your final data set.  

The other variables that you choose to include do not have to be collected by API or webscraped, but you do have to combine the files and clean the dataset with python.

Thus, you must submit:
- Your finalized data set (only one) (note: you may add more variables in the future).
- Your documented python notebook
- Any associated data files needed to produce the final dataset.

You will be evaluated on:
- Completeness of the data
- Quality of the code 
- The creativity of the new variable/webscraped data you gather

Be sure to upload ALL associated files for your code to run. I will run your code from beginning to end - make it easy for me to replicate your code.

In [336]:
import requests
import numpy as np
import pandas as pd
import pprint
from bs4 import BeautifulSoup

key = '6e5d84b217d445159bf25d5b7786d89f' 
fixed = 'https://api.twelvedata.com/' # switched to twelvedata because finnhub did not have historical data (only most recent)

mag7 = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA'] # ticker symbols for each of the mag7
lis=[] # making an empty list for later

for ticker in mag7:
    url=(fixed+
         'time_series?' # asking for type of data
         'start_date=2024-01-01' # beginning date of 2024
         '&symbol='
         + ticker + # pulls from the mag7 list
         '&interval=1day' # how often the data is being pulled
         '&outputsize=1000' # how much data is pulled
         '&apikey='
         + key) # API key
    response=requests.get(url) # GET request using the URL above
    data = response.json() # puts the data into json format
    df = pd.DataFrame(data['values']) # pulls the data under 'values' and puts it into a data frame
    volume = df[['datetime', 'volume']] # from the data frame, make a new data frame that only has the data and volume data
    volume.columns='Date',ticker # rename the columns into just the date of the information and the ticker symbol name
    lis.append(volume) # add the newly renamed dataframe into the list

In [164]:
lis[0] # testing

,Date,AAPL
0,2026-04-07,51070515
1,2026-04-06,29319825
2,2026-04-02,31289400
3,2026-04-01,40059400
4,2026-03-31,49598100
...,...,...
562,2024-01-08,59144500
563,2024-01-05,62303300
564,2024-01-04,71983600
565,2024-01-03,58414500


In [338]:
mag_volume=pd.DataFrame() # creating an empty dataframe to start
c=0 # counter
for ticker in mag7:
    ind=lis[c].set_index('Date') # creating another dataframe with the index being the date; picking the first 'lis' then the second and so on
    mag_volume=pd.concat([mag_volume,ind],axis=1) # combining the (initially) empty dataframe and the new dataframe; axis=1 so it all combines neatly
    c+=1 # progresses the counter
mag_volume

,AAPL,MSFT,GOOGL,AMZN,NVDA,META,TSLA
Date,,,,,,,
2026-04-07,60820961,20486199,20742991,26704731,124981494,8984804,68823279
2026-04-06,29329900,16146600,16945500,25385500,107564300,9515700,77697600
2026-04-02,31289400,24099100,21666500,31453000,143143200,13518500,83031200
2026-04-01,40059400,29417200,37684500,45955400,168132000,23608100,58683800
2026-03-31,49598100,45244400,43875400,58470100,226181300,32898300,75534900
...,...,...,...,...,...,...,...
2024-01-08,59144500,23134000,21404000,46757100,642510000,13890200,85166600
2024-01-05,62303300,20987000,22506000,45124800,415039000,13920700,92379400
2024-01-04,71983600,20901500,27137700,56039800,306535000,12099900,102629300


In [374]:
# !pip install yfinance

import yfinance as yf # https://pypi.org/project/yfinance/
vix = yf.Ticker("^VIX") # looks into the yahoo finance for whatever ticker we need
vix = vix.history(period="3y") # looks however far into the past we want
vix

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2023-04-10 00:00:00-05:00,19.389999,20.049999,18.930000,18.969999,0,0.0,0.0
2023-04-11 00:00:00-05:00,19.080000,19.280001,18.559999,19.100000,0,0.0,0.0
2023-04-12 00:00:00-05:00,19.379999,19.980000,18.250000,19.090000,0,0.0,0.0
2023-04-13 00:00:00-05:00,18.830000,19.059999,17.770000,17.799999,0,0.0,0.0
2023-04-14 00:00:00-05:00,17.940001,18.120001,17.070000,17.070000,0,0.0,0.0
...,...,...,...,...,...,...,...
2026-03-31 00:00:00-05:00,29.530001,29.639999,25.080000,25.250000,0,0.0,0.0
2026-04-01 00:00:00-05:00,24.299999,25.350000,23.500000,24.540001,0,0.0,0.0
2026-04-02 00:00:00-05:00,26.780001,27.889999,23.870001,23.870001,0,0.0,0.0


In [372]:
mag_volume.index = pd.to_datetime(mag_volume.index).date # turning both indices to datetime to merge cleanly https://stackoverflow.com/questions/16176996/keep-only-date-part-when-using-pandas-to-datetime
vix.index = pd.to_datetime(vix.index).date # turning both indices to datetime to merge cleanly. Had to add .date to remove the time.
mag_volume.index.name='Date'
vix.index.name='Date'

In [373]:
complete=pd.merge(mag_volume,vix,left_on=['Date'],right_on=['Date'],how='left') # merges based on the date of the mag_volume dataset
complete

,AAPL,MSFT,GOOGL,AMZN,NVDA,META,TSLA,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,,,,,,,,
2026-04-07,60820961,20486199,20742991,26704731,124981494,8984804,68823279,25.090000,28.000000,24.340000,25.780001,0,0.0,0.0
2026-04-06,29329900,16146600,16945500,25385500,107564300,9515700,77697600,24.930000,25.299999,23.780001,24.170000,0,0.0,0.0
2026-04-02,31289400,24099100,21666500,31453000,143143200,13518500,83031200,26.780001,27.889999,23.870001,23.870001,0,0.0,0.0
2026-04-01,40059400,29417200,37684500,45955400,168132000,23608100,58683800,24.299999,25.350000,23.500000,24.540001,0,0.0,0.0
2026-03-31,49598100,45244400,43875400,58470100,226181300,32898300,75534900,29.530001,29.639999,25.080000,25.250000,0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-01-08,59144500,23134000,21404000,46757100,642510000,13890200,85166600,14.000000,14.180000,13.020000,13.080000,0,0.0,0.0
2024-01-05,62303300,20987000,22506000,45124800,415039000,13920700,92379400,14.240000,14.580000,13.290000,13.350000,0,0.0,0.0
2024-01-04,71983600,20901500,27137700,56039800,306535000,12099900,102629300,13.930000,14.200000,13.640000,14.130000,0,0.0,0.0


Will be running a regression with the volume data of each of the stocks and the opening VIX price.

## ANYTHING FROM HERE IS EXPERIMENTAL

In [358]:
huh=pd.concat([mag_volume,vixt],axis=0)
huh

,AAPL,MSFT,GOOGL,AMZN,NVDA,META,TSLA,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,,,,,,,,
2026-04-07,60820961,20486199,20742991,26704731,124981494,8984804,68823279,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-04-06,29329900,16146600,16945500,25385500,107564300,9515700,77697600,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-04-02,31289400,24099100,21666500,31453000,143143200,13518500,83031200,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-04-01,40059400,29417200,37684500,45955400,168132000,23608100,58683800,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-03-31,49598100,45244400,43875400,58470100,226181300,32898300,75534900,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29.530001,29.639999,25.080000,25.250000,0.0,0.0,0.0
2026-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.299999,25.350000,23.500000,24.540001,0.0,0.0,0.0
2026-04-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.780001,27.889999,23.870001,23.870001,0.0,0.0,0.0


In [81]:
import requests
import numpy as np
import pandas as pd
import pprint
from bs4 import BeautifulSoup

key = 'IMP9CCQLGUUZT8WH' 
fixed = 'https://www.alphavantage.co/' # switched to twelvedata because finnhub did not have historical data (only most recent)

mag7 = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA'] # ticker symbols for each of the mag7
lis=[] # making an empty list for later

for ticker in mag7:
    url=(f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={ticker}&interval=weekly&apikey=IMP9CCQLGUUZT8WH')
    response=requests.get(url) # GET request using the URL above
    data = response.json() # puts the data into json format
    df=pd.DataFrame(data['Time Series (Daily)']).transpose() 
    volume = df['5. volume']
    volume.columns=ticker
    lis.append(volume) # add the newly renamed dataframe into the list

KeyError: 'Time Series (Daily)'

In [25]:
key = 'HJMJF9J88UKQHI2E' 
fixed = 'https://www.alphavantage.co/'

url=(fixed+
         'query?'
         'function=TIME_SERIES_DAILY'
         '&symbol='
         'TSLA'
         '&interval=weekly'
         '&apikey='
         + key) # API key
response=requests.get(url) # GET request using the URL above
data=response.json()

In [183]:
vix = requests.get('https://api.finage.co.uk/agg/index/VIX/1day/2024-01-01/2026-04-07?apikey=API_KEY29alGahyRFX2EB8W7wv9GgUKmtKdVwu5vUcW6FrJiM4XLi')
vix

<Response [401]>

In [185]:
import requests
import numpy as np
import pandas as pd
import pprint
from bs4 import BeautifulSoup

vixdata=vix.json()
vixdata

{'error': 'You do not have permission to make a request to this endpoint.'}